# Train the PCB defect detector (YOLOv8n) on Google Colab

**Project:** Bharat AI Hackathon, on-device PCB inspection on Raspberry Pi 5.

**What this notebook does:** it teaches a small YOLO model to find 6 kinds of PCB defect
(missing_hole, mouse_bite, open_circuit, short, spur, spurious_copper), measures how good it is,
and gives you a zip file with the trained model to copy to the Raspberry Pi.

**Before you start:** in the menu choose **Runtime → Change runtime type → T4 GPU → Save**.
Then run the cells one by one from top to bottom (click a cell and press **Shift + Enter**).
Keep this browser tab open until the last cell finishes.

## Step 1: Check that the GPU is on
Training on a GPU takes minutes; on a CPU it would take hours.
This cell should print a table that mentions **Tesla T4** (or another GPU).
If it prints an error instead, set the runtime type to T4 GPU (see above) and run it again.

In [ ]:
!nvidia-smi

## Step 2: Install the libraries
- `ultralytics` is the library that contains YOLO (training, testing, exporting).
- `roboflow` downloads the PCB dataset.

`-q` just means "quiet", so the output stays short.

In [ ]:
!pip install -q ultralytics roboflow

## Step 3: Download the PCB dataset
The dataset lives on Roboflow, which needs a free API key (a personal password for downloads):

1. Go to https://app.roboflow.com and sign in (Google sign-in works).
2. Open **Settings → API Keys** and copy the **Private API Key**.
3. Run the cell below and paste the key into the box that appears.

`getpass` hides what you paste, so the key is not saved inside the notebook.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

api_key = getpass("Paste your Roboflow API key and press Enter: ")

rf = Roboflow(api_key=api_key)
project = rf.workspace("jr-mqqnk").project("pcb-defects-detection-anddl")
dataset = project.version(1).download("yolov8")   # "yolov8" = the label format YOLO expects

print("Dataset saved in:", dataset.location)

## Step 4: Fix the dataset's settings file and count the images
`data.yaml` tells YOLO where the images are and what the classes are called.
The copy from Roboflow uses folder paths that often confuse YOLO, so we rewrite them
with the full, exact location. Then we count the images in each part:

- **train**: images the model learns from
- **valid**: images used during training to check progress
- **test**: images kept aside for the final, honest score

In [ ]:
import os, glob, yaml

yaml_path = os.path.join(dataset.location, "data.yaml")
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg["path"] = dataset.location
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
has_test = os.path.isdir(os.path.join(dataset.location, "test", "images"))
if has_test:
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)

with open(yaml_path, "w") as f:
    yaml.safe_dump(cfg, f)

print("Classes:", cfg["names"])
for split in ["train", "valid", "test"]:
    n = len(glob.glob(os.path.join(dataset.location, split, "images", "*")))
    print(f"{split:5s}: {n} images")

## Step 5: Train
- `YOLO("yolov8n.pt")` loads the smallest YOLOv8 model ("n" = nano), already trained on everyday
  objects. We continue its training on PCB defects, which is called **fine-tuning**.
- `epochs=50`: the model looks at the whole training set 50 times.
- `imgsz=640`: images are resized to 640 pixels before going into the model.
- `batch=16`: it studies 16 images at a time.

This is the slow cell (roughly 15 to 40 minutes). You will see one line per epoch.
The columns `P`, `R` and `mAP50` should slowly rise.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,                 # 0 = the first GPU
    project="/content/runs",
    name="pcb_v1",
    exist_ok=True,
)

save_dir = str(model.trainer.save_dir)
print("Training results saved in:", save_dir)

## Step 6: Measure how good the model is
We test the best version of the model on images it has never seen. These are the numbers for the report:

- **Precision**: of the defects it reported, how many were real (low precision = false alarms).
- **Recall**: of the real defects, how many it found (low recall = missed defects).
- **mAP50**: one overall score between 0 and 1 that combines both.

In [ ]:
best = YOLO(os.path.join(save_dir, "weights", "best.pt"))
split = "test" if has_test else "val"
metrics = best.val(data=yaml_path, split=split, imgsz=640)

summary = (
    f"Evaluated on the '{split}' images\n"
    f"Precision : {metrics.box.mp:.3f}\n"
    f"Recall    : {metrics.box.mr:.3f}\n"
    f"mAP50     : {metrics.box.map50:.3f}\n"
    f"mAP50-95  : {metrics.box.map:.3f}\n"
)
print(summary)

## Step 7: Convert the model for the Raspberry Pi
The Pi runs the model with ONNX Runtime, so we convert `best.pt` (PyTorch format)
into `best.onnx` (ONNX format). The model itself is the same; only the file format changes.

In [ ]:
onnx_path = best.export(format="onnx", imgsz=640)
print("ONNX model:", onnx_path)

## Step 8: Pack everything into one zip and download it
The zip contains:
- `yolo_pcb.onnx`: the model for the Raspberry Pi
- `yolo_pcb.pt`: the same model in PyTorch format (keep it, needed for any later re-training)
- `metrics.txt`: the scores from Step 6
- the training charts (`results.png`, confusion matrix, precision/recall curves) for the report
- `sample_images/`: 12 test PCB images with defects, to try in the dashboard

Your browser will download `pcb_model.zip` when this cell finishes.

In [ ]:
import shutil
from google.colab import files

out = "/content/pcb_model"
shutil.rmtree(out, ignore_errors=True)
os.makedirs(os.path.join(out, "sample_images"))

shutil.copy(onnx_path, os.path.join(out, "yolo_pcb.onnx"))
shutil.copy(os.path.join(save_dir, "weights", "best.pt"), os.path.join(out, "yolo_pcb.pt"))

with open(os.path.join(out, "metrics.txt"), "w") as f:
    f.write(summary)

for pattern in ["*.png", "*.jpg", "results.csv"]:
    for p in glob.glob(os.path.join(save_dir, pattern)):
        shutil.copy(p, out)

sample_split = "test" if has_test else "valid"
samples = sorted(glob.glob(os.path.join(dataset.location, sample_split, "images", "*")))[:12]
for p in samples:
    shutil.copy(p, os.path.join(out, "sample_images"))

shutil.make_archive("/content/pcb_model", "zip", out)
files.download("/content/pcb_model.zip")